# Test avec GradientBoostingRegressor

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
df = pd.read_csv("../data/gold.csv")

In [3]:
# column_titles = df.columns.tolist()
# column_titles

In [4]:
X = df.drop(["prix_median"], axis=1)
y = df['prix_median']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

categorial_features = ["proximité_autoroute"]

numerical_features = ['tx_crim', 'tx_residence','tx_commerce',
       'tx_nitriq', 'nb_piece', 'tx_ancienneté_parc_immo', 'distance_centre_emploi',
        'indice_impot_foncier',  'ratio_eleve_enseignant', 'tx_status_sociaux_eco_inf']

categorical_transformer = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorial_features),
        ('num', numerical_transformer, numerical_features)
    ],
    remainder="passthrough" 
)

# Créer un objet GradientBoostingRegressor
# model = GradientBoostingRegressor(random_state=42, verbose=False)
# model = GradientBoostingRegressor(n_estimators = 900, learning_rate = 0.09155068852731214, max_depth = 6, min_samples_split = 2, subsample = 0.8, loss = 'huber', criterion = 'squared_error', max_features = 1.0, random_state=42, verbose=False)
model = GradientBoostingRegressor(n_estimators = 600, learning_rate = 0.021825064275610876, max_depth = 7, min_samples_split = 6, subsample = 1.0, loss = 'huber', criterion = 'squared_error', max_features = 'sqrt', random_state=42, verbose=False)


# Créer un pipeline avec le préprocesseur et le modèle GradientBoostingRegressor
pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', model)
])

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :",mean_squared_error(y_train, y_pred_train))
print("TEST :",mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :",mean_absolute_error(y_train, y_pred_train))
print("TEST :",mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")    
print("TRAIN :",r2_score(y_train, y_pred_train))
print("TEST :",r2_score(y_test, y_pred_test))

Mean squared error : 
TRAIN : 0.05549775840283384
TEST : 7.259684146135703
Mean absolute error : 
TRAIN : 0.12909850009156432
TEST : 1.7900912239549653
R2 score : 
TRAIN : 0.9993611651442881
TEST : 0.901004914334441


In [5]:
# with mlflow.start_run():
#     mlflow.log_param("model", best_model.named_steps['model'].__class__.__name__)
#     mlflow.log_param("n_estimators", best_model.named_steps['model'].n_estimators)
#     mlflow.log_param("max_depth", best_model.named_steps['model'].max_depth)
#     mlflow.log_metric("mae", mae)
#     mlflow.log_metric("rmse", rmse)
#     mlflow.log_metric("r2", r2)
#     mlflow.sklearn.log_model(best_model, "model")

### Optuna (recherche des hyperparametres)

In [6]:
import optuna

In [7]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    n_estimators = trial.suggest_int('n_estimators', 100, 1000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    subsample = trial.suggest_float('subsample', 0.1, 1.0, step=0.1)
    loss = trial.suggest_categorical('loss', ['absolute_error', 'squared_error', 'huber', 'quantile'])
    criterion = trial.suggest_categorical('criterion', ['squared_error', 'friedman_mse'])
    max_features = trial.suggest_categorical('max_features', [1.0, 'sqrt', 'log2'])
    
    # Création du modèle avec les hyperparamètres suggérés
    model = GradientBoostingRegressor(n_estimators=n_estimators,
                                      learning_rate=learning_rate,
                                      max_depth=max_depth,
                                      min_samples_split=min_samples_split,
                                      subsample=subsample,
                                      loss=loss,
                                      criterion=criterion,
                                      max_features=max_features,
                                      random_state=42)
    
    # Même pipeline (préprocesseur + modèle) que pour l'évaluation finale
    pipe = Pipeline([
         ('preprocessor', preprocessor),
         ('model', model)
    ])
    
    # Validation croisée sur le jeu d'entraînement uniquement :
    # le jeu de test reste réservé à l'évaluation finale
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1).mean()
    
    return r2


In [8]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-10-08 11:46:43,532] A new study created in memory with name: no-name-2c7a707c-ec9f-4cc7-8e49-27ee1f3bfaf7


[I 2026-10-08 11:46:49,246] Trial 0 finished with value: 0.7735277334541633 and parameters: {'n_estimators': 800, 'learning_rate': 0.0036068603072975072, 'max_depth': 7, 'min_samples_split': 2, 'subsample': 0.30000000000000004, 'loss': 'absolute_error', 'criterion': 'friedman_mse', 'max_features': 1.0}. Best is trial 0 with value: 0.7735277334541633.


[I 2026-10-08 11:46:49,728] Trial 1 finished with value: 0.8344865622837163 and parameters: {'n_estimators': 300, 'learning_rate': 0.008940896442558296, 'max_depth': 6, 'min_samples_split': 3, 'subsample': 0.2, 'loss': 'squared_error', 'criterion': 'squared_error', 'max_features': 1.0}. Best is trial 1 with value: 0.8344865622837163.


[I 2026-10-08 11:46:51,972] Trial 2 finished with value: 0.8412817271605999 and parameters: {'n_estimators': 800, 'learning_rate': 0.03313133079992565, 'max_depth': 3, 'min_samples_split': 7, 'subsample': 0.2, 'loss': 'absolute_error', 'criterion': 'friedman_mse', 'max_features': 'log2'}. Best is trial 2 with value: 0.8412817271605999.


[I 2026-10-08 11:46:53,146] Trial 3 finished with value: 0.8182411143365502 and parameters: {'n_estimators': 200, 'learning_rate': 0.020922586160245216, 'max_depth': 4, 'min_samples_split': 6, 'subsample': 0.1, 'loss': 'huber', 'criterion': 'squared_error', 'max_features': 1.0}. Best is trial 2 with value: 0.8412817271605999.


...

[I 2026-10-08 11:58:19,490] Trial 95 finished with value: 0.8758988076164057 and parameters: {'n_estimators': 600, 'learning_rate': 0.06837552052881171, 'max_depth': 5, 'min_samples_split': 5, 'subsample': 1.0, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 59 with value: 0.8857535272286947.


[I 2026-10-08 11:58:24,805] Trial 96 finished with value: 0.8705072670151317 and parameters: {'n_estimators': 700, 'learning_rate': 0.07521641266981331, 'max_depth': 5, 'min_samples_split': 5, 'subsample': 0.9, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 59 with value: 0.8857535272286947.


[I 2026-10-08 11:58:29,898] Trial 97 finished with value: 0.8741844369004419 and parameters: {'n_estimators': 700, 'learning_rate': 0.05025213905384525, 'max_depth': 5, 'min_samples_split': 5, 'subsample': 0.9, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 59 with value: 0.8857535272286947.


[I 2026-10-08 11:58:34,854] Trial 98 finished with value: 0.8839682593561398 and parameters: {'n_estimators': 600, 'learning_rate': 0.08545144559317534, 'max_depth': 5, 'min_samples_split': 4, 'subsample': 1.0, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 59 with value: 0.8857535272286947.


[I 2026-10-08 11:58:41,710] Trial 99 finished with value: 0.8789454663082555 and parameters: {'n_estimators': 700, 'learning_rate': 0.07547895746165004, 'max_depth': 6, 'min_samples_split': 6, 'subsample': 1.0, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}. Best is trial 59 with value: 0.8857535272286947.


In [9]:
best_params = study.best_params
best_value = study.best_value

In [10]:
print(best_params)
print(best_value)

{'n_estimators': 600, 'learning_rate': 0.035788649532663336, 'max_depth': 5, 'min_samples_split': 5, 'subsample': 1.0, 'loss': 'huber', 'criterion': 'friedman_mse', 'max_features': 'sqrt'}
0.8857535272286947


### Évaluation finale sur le jeu de test

In [11]:
# Pipeline avec les meilleurs hyperparamètres trouvés par Optuna,
# entraîné sur le jeu d'entraînement puis évalué une seule fois sur le jeu de test
best_pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', GradientBoostingRegressor(**best_params, random_state=42))
])
best_pipe.fit(X_train, y_train)
y_pred_test = best_pipe.predict(X_test)

print("TEST MSE :", mean_squared_error(y_test, y_pred_test))
print("TEST MAE :", mean_absolute_error(y_test, y_pred_test))
print("TEST R2 :", r2_score(y_test, y_pred_test))

TEST MSE : 7.035060472433613
TEST MAE : 1.8006305206455375
TEST R2 : 0.904067945641731
